In [1]:
import pandas as pd
import numpy as np

In [3]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [5]:
df = pd.read_csv('covid_toy.csv')

In [7]:
df.sample(10)

,age,gender,fever,cough,city,has_covid
1,27,Male,100.0,Mild,Delhi,Yes
5,84,Female,NaN,Mild,Bangalore,Yes
97,20,Female,101.0,Mild,Bangalore,No
15,70,Male,103.0,Strong,Kolkata,Yes
3,31,Female,98.0,Mild,Kolkata,No
10,75,Female,NaN,Mild,Delhi,No
7,20,Female,NaN,Strong,Mumbai,Yes
91,38,Male,NaN,Mild,Delhi,Yes
26,19,Female,100.0,Mild,Kolkata,Yes
58,23,Male,98.0,Strong,Mumbai,Yes


In [13]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [31]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [33]:
X_train

,age,gender,fever,cough,city
24,13,Female,100.0,Strong,Kolkata
76,80,Male,100.0,Mild,Bangalore
28,16,Male,104.0,Mild,Kolkata
3,31,Female,98.0,Mild,Kolkata
16,69,Female,103.0,Mild,Kolkata
...,...,...,...,...,...
81,65,Male,99.0,Mild,Delhi
47,18,Female,104.0,Mild,Bangalore
6,14,Male,101.0,Strong,Bangalore
56,71,Male,NaN,Strong,Kolkata


In [35]:
y_train

24     No
76    Yes
28     No
3      No
16    Yes
     ... 
81     No
47     No
6      No
56     No
31     No
Name: has_covid, Length: 80, dtype: object

## aam zindagi

In [38]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [40]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [42]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


(80, 4)

In [44]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [46]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## mentos zindagi

In [59]:
from sklearn.compose import ColumnTransformer

In [71]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse=False,drop='first'),['gender','city'])]
    ,remainder='passthrough')

In [73]:
transformer.fit_transform(X_train).shape

/opt/anaconda3/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


(80, 7)

In [75]:
transformer.transform(X_test).shape

(20, 7)